# Voice cloning tiếng Việt trên Colab

Notebook này tạo file MP3 mới dựa trên giọng trong `2.mp3`. Chỉ dùng với giọng của chính bạn hoặc khi bạn có quyền/sự đồng ý rõ ràng từ người sở hữu giọng nói.

**Cách chạy:** Runtime > Change runtime type > GPU, sau đó chạy lần lượt từng cell. Khi được hỏi upload file, chọn đúng `2.mp3`.


Repo: https://github.com/k2-fsa/OmniVoice

OmniVoice là zero-shot voice cloning TTS hỗ trợ 600+ ngôn ngữ. Notebook này dùng reference `2.mp3`, tự transcribe bằng Whisper để điền `ref_text`, synthesize tiếng Việt rồi xuất MP3.

In [ ]:
TEXT_TO_SPEAK = """Xin chào, đây là bản thử nghiệm nhân bản giọng nói tiếng Việt. Nếu đoạn âm thanh tham chiếu rõ ràng, ít nhiễu và chỉ có một người nói, kết quả sẽ giống giọng gốc hơn."""
STYLE = "tu_nhien"  # dùng cho một số model: tu_nhien, tin_tuc, doc_truyen
REFERENCE_MP3 = "/content/2.mp3"
REFERENCE_WAV = "/content/ref.wav"

OUTPUT_WAV = "/content/omnivoice_output.wav"
OUTPUT_TRIMMED_WAV = "/content/omnivoice_output_trimmed.wav"
OUTPUT_MP3 = "/content/omnivoice_output.mp3"
LANGUAGE_ID = "vi"
NUM_STEP = 16  # 16 nhanh hơn, 32 thường ổn định/chất lượng hơn nhưng chậm hơn.
SPEED = 1.0

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg

# OmniVoice README khuyến nghị torch/torchaudio 2.8.0 CUDA 12.8.
!pip -q install --force-reinstall --no-deps torch==2.8.0+cu128 torchaudio==2.8.0+cu128 torchvision==0.23.0+cu128 --index-url https://download.pytorch.org/whl/cu128
!pip -q install -U git+https://github.com/k2-fsa/OmniVoice.git faster-whisper soundfile

In [ ]:
from google.colab import files
import os, shutil, subprocess, textwrap

uploaded = files.upload()
uploaded_name = "2.mp3" if "2.mp3" in uploaded else next(iter(uploaded))
uploaded_path = os.path.abspath(uploaded_name)
if uploaded_path != REFERENCE_MP3:
    shutil.move(uploaded_path, REFERENCE_MP3)

assert os.path.exists(REFERENCE_MP3), "Không tìm thấy /content/2.mp3"

# Cắt 12 giây đầu, mono 24kHz. Nếu file gốc có đoạn im lặng đầu, hãy đổi -ss 0 thành vị trí bắt đầu giọng nói.
subprocess.run([
    "ffmpeg", "-y", "-i", REFERENCE_MP3,
    "-ss", "0", "-t", "12",
    "-ar", "24000", "-ac", "1",
    REFERENCE_WAV
], check=True)

print("Reference WAV:", REFERENCE_WAV)

In [ ]:
import torch, os
from faster_whisper import WhisperModel

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"
asr = WhisperModel("small", device=device, compute_type=compute_type)
segments, info = asr.transcribe(REFERENCE_WAV, language="vi", vad_filter=True)
REF_TEXT = " ".join(seg.text.strip() for seg in segments).strip()
print("REF_TEXT =", REF_TEXT)
if not REF_TEXT:
    raise RuntimeError("Whisper không nhận ra transcript từ 2.mp3. Hãy dùng đoạn ref rõ tiếng hơn.")

In [ ]:
import gc, subprocess, os
import torch
import soundfile as sf
from omnivoice import OmniVoice

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device_map = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map=device_map,
    dtype=dtype,
)

generate_kwargs = dict(
    text=TEXT_TO_SPEAK.strip(),
    ref_audio=REFERENCE_WAV,
    ref_text=REF_TEXT.strip(),
    num_step=NUM_STEP,
    speed=SPEED,
)

# Một số version hỗ trợ language_id, một số version tự detect ngôn ngữ từ text.
try:
    audio = model.generate(**generate_kwargs, language_id=LANGUAGE_ID)
except TypeError as exc:
    print("API không nhận language_id, chạy lại không truyền language_id:", exc)
    audio = model.generate(**generate_kwargs)

wav = audio[0] if isinstance(audio, (list, tuple)) else audio
sf.write(OUTPUT_WAV, wav, 24000)

# Trim silence nhẹ ở đầu/cuối, giữ nguyên nội dung lời nói.
subprocess.run([
    "ffmpeg", "-y", "-i", OUTPUT_WAV,
    "-af", "silenceremove=start_periods=1:start_duration=0.15:start_threshold=-45dB:stop_periods=1:stop_duration=0.35:stop_threshold=-45dB",
    OUTPUT_TRIMMED_WAV,
], check=True)

subprocess.run(["ffmpeg", "-y", "-i", OUTPUT_TRIMMED_WAV, "-codec:a", "libmp3lame", "-q:a", "2", OUTPUT_MP3], check=True)
print("Done:", OUTPUT_MP3)

In [ ]:
from google.colab import files
files.download(OUTPUT_MP3)